# AWS Glue Notebook; Crypto Data Lake Pipeline

This notebook processes cryptocurrency market data stored in Amazon S3 using AWS Glue and PySpark.

## Pipeline overview
- **Bronze layer**: raw CSV files ingested into S3
- **Silver layer**: cleaned and partitioned Parquet datasets
- **Gold layer**: technical indicators and KPI-ready datasets

## Main outputs
- Silver datasets partitioned by `symbol` and `year`
- Gold datasets with technical indicators such as SMA, EMA, RSI, and MACD


## 1. Glue interactive session setup

Run the following cell in AWS Glue Studio to start the interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)


## 2. Imports and configuration

Define source and destination paths and load the Spark functions used throughout the notebook.


In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import avg, col, lag, substring, to_timestamp, when
import pyspark.sql.functions as F

BRONZE_SOURCE_PATH = "s3://cargadatostradingview/*/*/*.csv"
SILVER_DESTINATION_PATH = "s3://cargadatosplata/"
GOLD_DESTINATION_PATH = "s3://cargadatosoro/"


## 3. Read the bronze layer

Load the raw CSV files from S3 and inspect the schema.


In [ ]:
bronze_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(BRONZE_SOURCE_PATH)
)

bronze_df.printSchema()
bronze_df.show(10, truncate=False)


## 4. Build the silver layer

Clean the raw dataset, standardize data types, extract the year from the datetime field, and store the result as partitioned Parquet files.


In [ ]:
silver_df = (
    bronze_df
    .withColumn("datetime", to_timestamp(col("datetime")))
    .withColumn("open", col("open").cast("double"))
    .withColumn("high", col("high").cast("double"))
    .withColumn("low", col("low").cast("double"))
    .withColumn("close", col("close").cast("double"))
    .withColumn("volume", col("volume").cast("double"))
    .withColumn("year", substring(col("datetime").cast("string"), 1, 4))
)

silver_df.show(10, truncate=False)

(
    silver_df.write
    .mode("append")
    .partitionBy("symbol", "year")
    .parquet(SILVER_DESTINATION_PATH)
)

print("Silver layer stored successfully in S3.")


## 5. KPI functions for the gold layer

The following functions compute technical indicators for each symbol and year.


In [ ]:
def calculate_sma(df, period: int = 14):
    """
    Calculate the Simple Moving Average (SMA) over the close price.

    Args:
        df: Input Spark DataFrame containing at least symbol, year, datetime, and close.
        period: Number of previous rows used in the rolling average.

    Returns:
        Spark DataFrame with an additional SMA column.
    """
    window_spec = (
        Window.partitionBy("symbol", "year")
        .orderBy("datetime")
        .rowsBetween(-period, 0)
    )
    return df.withColumn(f"SMA_{period}", F.avg("close").over(window_spec))


def calculate_ema(df, period: int = 14):
    """
    Calculate an approximate Exponential Moving Average (EMA).

    Args:
        df: Input Spark DataFrame containing at least symbol, year, datetime, and close.
        period: Smoothing period for the EMA.

    Returns:
        Spark DataFrame with an additional EMA column.
    """
    alpha = 2 / (period + 1)
    window_spec = Window.partitionBy("symbol", "year").orderBy("datetime")

    df = df.withColumn("prev_close", lag("close", 1).over(window_spec))
    return df.withColumn(
        f"EMA_{period}",
        (alpha * col("close")) + ((1 - alpha) * col("prev_close"))
    )


def calculate_rsi(df, period: int = 14):
    """
    Calculate the Relative Strength Index (RSI).

    Args:
        df: Input Spark DataFrame containing at least symbol, year, datetime, and close.
        period: Window size used to compute average gains and losses.

    Returns:
        Spark DataFrame with an RSI column.
    """
    base_window = Window.partitionBy("symbol", "year").orderBy("datetime")
    rolling_window = base_window.rowsBetween(-period, 0)

    df = df.withColumn("change", col("close") - lag("close", 1).over(base_window))
    df = df.withColumn("gain", when(col("change") > 0, col("change")).otherwise(0.0))
    df = df.withColumn("loss", when(col("change") < 0, -col("change")).otherwise(0.0))

    avg_gain = avg("gain").over(rolling_window)
    avg_loss = avg("loss").over(rolling_window)

    return df.withColumn("RSI", 100 - (100 / (1 + (avg_gain / avg_loss))))


def calculate_macd(df, short_period: int = 12, long_period: int = 26, signal_period: int = 9):
    """
    Calculate the MACD indicator and its signal line.

    Args:
        df: Input Spark DataFrame containing at least symbol, year, datetime, and close.
        short_period: Period used for the short EMA.
        long_period: Period used for the long EMA.
        signal_period: Rolling window used for the signal line.

    Returns:
        Spark DataFrame with MACD and Signal_Line columns.
    """
    df = calculate_ema(df, short_period).withColumnRenamed(f"EMA_{short_period}", "short_ema")
    df = calculate_ema(df, long_period).withColumnRenamed(f"EMA_{long_period}", "long_ema")

    df = df.withColumn("MACD", col("short_ema") - col("long_ema"))

    signal_window = (
        Window.partitionBy("symbol", "year")
        .orderBy("datetime")
        .rowsBetween(-signal_period, 0)
    )
    return df.withColumn("Signal_Line", avg("MACD").over(signal_window))


## 6. Read the silver layer and compute KPIs

Load the cleaned silver data, compute indicators, and preview the result before storing the gold layer.


In [ ]:
silver_kpi_df = (
    spark.read
    .parquet(SILVER_DESTINATION_PATH)
    .withColumn("datetime", to_timestamp(col("datetime")))
    .withColumn("close", col("close").cast("double"))
)

gold_df = calculate_sma(silver_kpi_df, period=14)
gold_df = calculate_ema(gold_df, period=14)
gold_df = calculate_rsi(gold_df, period=14)
gold_df = calculate_macd(gold_df)

gold_preview_df = gold_df.select(
    "datetime",
    "symbol",
    "year",
    "close",
    "SMA_14",
    "EMA_14",
    "RSI",
    "MACD",
    "Signal_Line",
)

gold_preview_df.show(10, truncate=False)


## 7. Store the gold layer

Write the KPI-enriched dataset back to S3 in Parquet format.


In [ ]:
(
    gold_df.write
    .mode("append")
    .partitionBy("symbol", "year")
    .parquet(GOLD_DESTINATION_PATH)
)

print("Gold layer stored successfully in S3.")
